In [ ]:
```xml
<VSCode.Cell language="markdown">
# 🏢 Multi-Tenancy & SaaS Architecture Guide

**Building scalable, secure multi-tenant systems with proper isolation and resource management**

This guide covers:
- Multi-tenancy isolation levels
- Tenant data isolation strategies
- Resource quotas and entitlements
- Tenant provisioning
- Scaling multi-tenant systems
- Security isolation
- Billing and metering
- Tenant lifecycle management
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Tenant Isolation & Architecture

### Tenant Management
```python
from dataclasses import dataclass, field
from typing import Dict, List, Optional
from datetime import datetime
from enum import Enum

class IsolationLevel(Enum):
    """Data isolation levels"""
    SINGLE_DATABASE = "single_database"
    SCHEMA_ISOLATION = "schema_isolation"
    DATABASE_PER_TENANT = "database_per_tenant"

@dataclass
class TenantConfig:
    """Tenant configuration"""
    tenant_id: str
    tenant_name: str
    isolation_level: IsolationLevel
    database_name: str
    schema_name: Optional[str]
    max_users: int
    max_storage_gb: int
    api_rate_limit_per_minute: int
    features_enabled: List[str] = field(default_factory=list)
    created_at: datetime = field(default_factory=datetime.now)
    status: str = "active"

class TenantRegistry:
    """Registry of tenants"""
    
    def __init__(self):
        self.tenants: Dict[str, TenantConfig] = {}
        self.tenant_users: Dict[str, List[str]] = {}
        self.domain_to_tenant: Dict[str, str] = {}
    
    def register_tenant(self, config: TenantConfig) -> bool:
        """Register new tenant"""
        
        if config.tenant_id in self.tenants:
            return False
        
        self.tenants[config.tenant_id] = config
        self.tenant_users[config.tenant_id] = []
        
        return True
    
    def map_domain(self, domain: str, tenant_id: str) -> bool:
        """Map domain to tenant"""
        
        if tenant_id not in self.tenants:
            return False
        
        self.domain_to_tenant[domain] = tenant_id
        return True
    
    def get_tenant_by_domain(self, domain: str) -> Optional[TenantConfig]:
        """Get tenant by domain"""
        
        tenant_id = self.domain_to_tenant.get(domain)
        
        if not tenant_id:
            return None
        
        return self.tenants.get(tenant_id)
    
    def resolve_tenant_context(self, user_id: str) -> Optional[TenantConfig]:
        """Resolve tenant from user"""
        
        for tenant_id, users in self.tenant_users.items():
            if user_id in users:
                return self.tenants[tenant_id]
        
        return None
    
    def add_user_to_tenant(self, tenant_id: str, user_id: str) -> bool:
        """Add user to tenant"""
        
        if tenant_id not in self.tenants:
            return False
        
        if user_id not in self.tenant_users[tenant_id]:
            self.tenant_users[tenant_id].append(user_id)
        
        return True

class TenantIsolationController:
    """Control tenant isolation"""
    
    def __init__(self, isolation_level: IsolationLevel):
        self.isolation_level = isolation_level
    
    def get_database_connection_string(self, tenant_config: TenantConfig) -> str:
        """Get connection string based on isolation level"""
        
        if self.isolation_level == IsolationLevel.SINGLE_DATABASE:
            return f"postgresql://localhost/{tenant_config.database_name}"
        
        elif self.isolation_level == IsolationLevel.SCHEMA_ISOLATION:
            return f"postgresql://localhost/{tenant_config.database_name}?schema={tenant_config.schema_name}"
        
        elif self.isolation_level == IsolationLevel.DATABASE_PER_TENANT:
            return f"postgresql://localhost/{tenant_config.database_name}"
        
        return ""
    
    def get_query_filter(self, tenant_config: TenantConfig) -> str:
        """Get SQL filter for tenant isolation"""
        
        if self.isolation_level == IsolationLevel.SINGLE_DATABASE:
            return f"WHERE tenant_id = '{tenant_config.tenant_id}'"
        
        else:
            return ""  # Schema/database isolation handles this at connection level
```

### Row-Level Security
```python
class RowLevelSecurityPolicy:
    """Row-level security policy"""
    
    def __init__(self, table_name: str, tenant_column: str):
        self.table_name = table_name
        self.tenant_column = tenant_column
        self.enabled = True
    
    def get_policy_definition(self) -> str:
        """Get SQL policy definition"""
        
        policy_sql = f"""
        CREATE POLICY {self.table_name}_tenant_isolation ON {self.table_name}
        USING ({self.tenant_column} = current_setting('app.tenant_id'))
        WITH CHECK ({self.tenant_column} = current_setting('app.tenant_id'));
        """
        
        return policy_sql

class RLSManager:
    """Manage row-level security"""
    
    def __init__(self):
        self.policies: Dict[str, RowLevelSecurityPolicy] = {}
    
    def create_policy(self, table_name: str, tenant_column: str) -> bool:
        """Create RLS policy"""
        
        policy = RowLevelSecurityPolicy(table_name, tenant_column)
        self.policies[table_name] = policy
        
        return True
    
    def set_tenant_context(self, tenant_id: str) -> str:
        """Set tenant context for current session"""
        
        return f"SET app.tenant_id = '{tenant_id}';"
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Resource Management & Quotas

### Tenant Resource Quotas
```python
@dataclass
class ResourceQuota:
    """Resource quota for tenant"""
    tenant_id: str
    max_storage_gb: int
    max_concurrent_connections: int
    max_api_calls_per_minute: int
    max_users: int
    storage_used_gb: float = 0.0
    api_calls_this_minute: int = 0
    active_connections: int = 0

class QuotaManager:
    """Manage tenant quotas"""
    
    def __init__(self):
        self.quotas: Dict[str, ResourceQuota] = {}
        self.quota_violations: List[Dict] = []
    
    def set_quota(self, quota: ResourceQuota) -> bool:
        """Set tenant quota"""
        
        self.quotas[quota.tenant_id] = quota
        return True
    
    def check_storage_quota(self, tenant_id: str, additional_gb: float) -> bool:
        """Check if storage quota allows addition"""
        
        if tenant_id not in self.quotas:
            return False
        
        quota = self.quotas[tenant_id]
        
        if quota.storage_used_gb + additional_gb > quota.max_storage_gb:
            self.quota_violations.append({
                'tenant_id': tenant_id,
                'quota_type': 'storage',
                'timestamp': datetime.now()
            })
            return False
        
        return True
    
    def check_api_quota(self, tenant_id: str) -> bool:
        """Check if API quota allows request"""
        
        if tenant_id not in self.quotas:
            return False
        
        quota = self.quotas[tenant_id]
        
        if quota.api_calls_this_minute >= quota.max_api_calls_per_minute:
            self.quota_violations.append({
                'tenant_id': tenant_id,
                'quota_type': 'api',
                'timestamp': datetime.now()
            })
            return False
        
        quota.api_calls_this_minute += 1
        return True
    
    def check_connection_quota(self, tenant_id: str) -> bool:
        """Check if connection quota available"""
        
        if tenant_id not in self.quotas:
            return False
        
        quota = self.quotas[tenant_id]
        
        if quota.active_connections >= quota.max_concurrent_connections:
            return False
        
        quota.active_connections += 1
        return True
    
    def get_quota_utilization(self, tenant_id: str) -> Dict:
        """Get quota utilization"""
        
        if tenant_id not in self.quotas:
            return {}
        
        quota = self.quotas[tenant_id]
        
        return {
            'storage_percent': (quota.storage_used_gb / quota.max_storage_gb) * 100,
            'api_calls_percent': (quota.api_calls_this_minute / quota.max_api_calls_per_minute) * 100,
            'connections_percent': (quota.active_connections / quota.max_concurrent_connections) * 100
        }
```

### Metering & Billing
```python
class UsageMetrics:
    """Track usage metrics"""
    
    def __init__(self, tenant_id: str):
        self.tenant_id = tenant_id
        self.api_calls = 0
        self.storage_gb = 0.0
        self.active_users = 0
        self.compute_minutes = 0.0
    
    def record_api_call(self):
        """Record API call"""
        self.api_calls += 1
    
    def get_monthly_cost(self, pricing: Dict) -> float:
        """Calculate monthly cost"""
        
        cost = 0.0
        
        cost += (self.api_calls / 1000000) * pricing.get('api_calls_per_million', 1.0)
        cost += self.storage_gb * pricing.get('storage_per_gb', 0.10)
        cost += self.active_users * pricing.get('per_user', 10.0)
        cost += self.compute_minutes * pricing.get('compute_per_minute', 0.001)
        
        return cost

class BillingEngine:
    """Calculate and manage billing"""
    
    def __init__(self):
        self.usage_metrics: Dict[str, UsageMetrics] = {}
        self.pricing_tiers: Dict[str, Dict] = {}
        self.invoices: List[Dict] = []
    
    def record_usage(self, tenant_id: str, metric_type: str, amount: float):
        """Record usage for tenant"""
        
        if tenant_id not in self.usage_metrics:
            self.usage_metrics[tenant_id] = UsageMetrics(tenant_id)
        
        metrics = self.usage_metrics[tenant_id]
        
        if metric_type == "api_calls":
            metrics.api_calls += amount
        elif metric_type == "storage_gb":
            metrics.storage_gb += amount
        elif metric_type == "compute_minutes":
            metrics.compute_minutes += amount
    
    def generate_invoice(self, tenant_id: str, pricing: Dict) -> Optional[Dict]:
        """Generate invoice for tenant"""
        
        if tenant_id not in self.usage_metrics:
            return None
        
        metrics = self.usage_metrics[tenant_id]
        total_cost = metrics.get_monthly_cost(pricing)
        
        invoice = {
            'tenant_id': tenant_id,
            'period': datetime.now().strftime("%Y-%m"),
            'total_cost': total_cost,
            'usage_metrics': {
                'api_calls': metrics.api_calls,
                'storage_gb': metrics.storage_gb,
                'compute_minutes': metrics.compute_minutes
            },
            'created_at': datetime.now()
        }
        
        self.invoices.append(invoice)
        
        return invoice
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Tenant Provisioning & Lifecycle

### Tenant Provisioning
```python
class TenantProvisioningOrchestrator:
    """Orchestrate tenant provisioning"""
    
    def __init__(self):
        self.provisioning_steps: List[str] = [
            "create_database",
            "create_schema",
            "initialize_tables",
            "create_policies",
            "create_credentials",
            "configure_monitoring",
            "register_tenant"
        ]
        self.provisioning_status: Dict[str, Dict] = {}
    
    def provision_tenant(self, config: TenantConfig) -> bool:
        """Provision new tenant"""
        
        tenant_id = config.tenant_id
        self.provisioning_status[tenant_id] = {
            'status': 'in_progress',
            'completed_steps': [],
            'started_at': datetime.now()
        }
        
        for step in self.provisioning_steps:
            try:
                # Execute provisioning step
                self._execute_step(step, config)
                
                self.provisioning_status[tenant_id]['completed_steps'].append(step)
                
            except Exception as e:
                self.provisioning_status[tenant_id]['status'] = 'failed'
                self.provisioning_status[tenant_id]['error'] = str(e)
                return False
        
        self.provisioning_status[tenant_id]['status'] = 'completed'
        self.provisioning_status[tenant_id]['completed_at'] = datetime.now()
        
        return True
    
    def _execute_step(self, step: str, config: TenantConfig):
        """Execute individual provisioning step"""
        
        if step == "create_database":
            print(f"Creating database: {config.database_name}")
        elif step == "create_schema":
            print(f"Creating schema: {config.schema_name}")
        elif step == "initialize_tables":
            print("Initializing tables")
        # ... other steps
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. Multi-Tenancy Checklist

✅ **Architecture**
- [ ] Isolation level chosen
- [ ] Tenant registry implemented
- [ ] Domain mapping configured
- [ ] Connection pooling set up
- [ ] Request routing working

✅ **Security**
- [ ] Row-level security enabled
- [ ] Tenant context isolation
- [ ] Data encryption in transit
- [ ] Data encryption at rest
- [ ] Access controls verified

✅ **Resource Management**
- [ ] Quotas defined
- [ ] Rate limiting configured
- [ ] Storage limits enforced
- [ ] Connection limits enforced
- [ ] Monitoring active

✅ **Billing & Metering**
- [ ] Usage metrics tracked
- [ ] Pricing tiers defined
- [ ] Invoice generation
- [ ] Payment processing
- [ ] Reporting dashboard

✅ **Operational**
- [ ] Provisioning automated
- [ ] Tenant onboarding smooth
- [ ] Monitoring per-tenant
- [ ] Alerting configured
- [ ] Runbooks documented
</VSCode.Cell>
```